# Step B - Multiple Instance Detection

In addition to what was achieved in step A, the system should now be able to detect multiple instances of the same product within a single scene. As before, we are given one reference image per product and a scene image containing several products placed on the shelves. The goal is for the system to correctly identify all occurrences of each product in the scene.

We begin by importing the required libraries, defining the directories for template and scene images, listing the template IDs, and setting the thresholds for feature matching and color filtering, as well as visualization parameters.
In short, the initial setup is the same as in Step A, ensuring that Task B remains self-contained and independent from the previous implementation.
  
Then, we will reuse the two previously defined functions (`def preprocess_scene(img_scene)` and `def calc_mean_hue(img_bgr)`) and initialize `SIFT` and the `BFMatcher`.

In [1]:
import cv2
import numpy as np
import os

# Directories containing template (model) and scene (shelf) images
MODELS_DIR       = "./models/"    # contains files named 0.jpg, 1.jpg, ..., 26.jpg
SCENES_DIR       = './scenes/'    # contains files e1.png to e5.png

MODEL_IDS        = [0, 1, 11, 19, 24, 25, 26]
SCENE_FILES      = ["m1.png", "m2.png", "m3.png", "m4.png", "m5.png"]

# Models to be discriminated with color filter (HUE)
CONFUSE_MODELS   = {1, 11, 0, 26}
HUE_DIFF_THRESH  = 18  # Allowed Hue difference (degrees)

# Matching and filtering parameters
MIN_MATCHES         = 30     # minimum number of total matches required to consider the model
RATIO_TEST          = 0.7    # Lowe’s ratio threshold for the ratio test filter
RANSAC_THRESH       = 3.0    # RANSAC reprojection threshold in pixels for homography estimation
MIN_AREA_RATIO      = 0.01   # minimum detected area relative to the scene area (filter)
CLUSTER_EPS         = 30.0   # radius (in pixels) for GHT vote clustering
MIN_CLUSTER_INLIERS = 8      # minimum number of inliers required to accept an instance

# Visualization style for results
BOX_COLOR    = (0, 255, 0)   # green color for bounding boxes
CENTER_COLOR = (0, 0, 255)   # red color for centroids
TEXT_COLOR   = (0, 255, 0)   # text color (green)
FONT         = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE   = 0.7
TEXT_THICK   = 2
BOX_THICK    = 10
CIRCLE_RAD   = 4


In [2]:
def preprocess_scene(img_scene): 
    lab = cv2.cvtColor(img_scene, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def calc_mean_hue(img_bgr): 
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    return float(np.mean(hsv[:, :, 0]))

In [3]:
sift = cv2.SIFT_create()
bf   = cv2.BFMatcher(cv2.NORM_L2)

This block loads and preprocesses all reference models so they’re ready for matching in Step B.
For each `mid` it reads the model image, extracts SIFT keypoints/descriptors, records the image size, and computes a keypoint barycenter (used as a robust template center; falls back to the image center if no keypoints are found). For models listed in `CONFUSE_MODELS`, it also computes the mean Hue to enable a later color-based disambiguation. All this metadata is stored in the `models` dictionary (`img`, `kp`, `des`, `size`, `centroid`, `mean_hue`) and will be reused during scene matching and multi-instance detection.

In [4]:
# Model loading and preprocessing
models = {}
for mid in MODEL_IDS:
    path = os.path.join(MODELS_DIR, f"{mid}.jpg")
    img_model = cv2.imread(path)
    if img_model is None:
        raise FileNotFoundError(f"Model image not found: {path}")
    # Extract key points and descriptors from the model
    kp_model, des_model = sift.detectAndCompute(img_model, None)
    h_model, w_model = img_model.shape[:2]
    # Compute centroid (barycenter) of the model's keypoints
    if kp_model is not None and len(kp_model) > 0:
        pts = np.float32([kp.pt for kp in kp_model])
        cx_model = float(np.mean(pts[:,0]))
        cy_model = float(np.mean(pts[:,1]))
    else:
        # if there are no keypoints, use the center of the image as a fallback
        cx_model = w_model / 2.0
        cy_model = h_model / 2.0
    # Compute mean hue for "confused" models (to be filtered by color)
    mean_hue = None
    if mid in CONFUSE_MODELS:
        mean_hue = calc_mean_hue(img_model)
    # Save model data
    models[mid] = {
        'img': img_model,
        'kp': kp_model,
        'des': des_model,
        'size': (w_model, h_model),
        'centroid': (cx_model, cy_model),
        'mean_hue': mean_hue
    }

What this following block does.    
For each scene image, it finds all occurrences of each product by:

1. extracting scene features,
2. matching them to each model’s features,
3. voting (GHT-style) for likely centroid positions of the product,
4. clustering votes to separate multiple instances,
5. verifying each cluster geometrically via homography + RANSAC,
6. refining with area and color filters, and
7. recording and visualizing the detections.

In [5]:
# Process each scene image
for scene_file in SCENE_FILES:
    scene_path = os.path.join(SCENES_DIR, scene_file)
    img_scene = cv2.imread(scene_path)
    if img_scene is None:
        raise FileNotFoundError(f"Scene image not found: {scene_path}")
    # Pre-processing of the scene
    img_proc = preprocess_scene(img_scene)
    h_scene, w_scene = img_proc.shape[:2]
    # Extract keypoints and descriptors from the scene
    kp_scene, des_scene = sift.detectAndCompute(img_proc, None)
    print(f"\n=== Scene Analysis {scene_file} ===")
    print(f"Kp scena: {len(kp_scene)}  Descriptors: {None if des_scene is None else des_scene.shape}")
    # Dictionary for detections found in the current scene
    detections = {}
    # For each model, perform matching and Generalized Hough Transform to detect instances
    for mid, data in models.items():
        kp_model = data['kp']
        des_model = data['des']
        w_model, h_model = data['size']
        cx_model, cy_model = data['centroid']
        print(f"\n[Modello {mid}]: kp_modello={len(kp_model)}  des_modello={None if des_model is None else des_model.shape}")
        if des_model is None or des_scene is None:
            print("  => Skip: descriptors missing")
            continue
        # 1) Matching SIFT with Lowe's ratio test
        matches = bf.knnMatch(des_model, des_scene, k=2)
        print(f"  Raw matches: {len(matches)}")
        good = [m for m, n in matches if m.distance < RATIO_TEST * n.distance]
        print(f"  Good matches (ratio<{RATIO_TEST}): {len(good)}")
        if len(good) < MIN_MATCHES:
            print(f"  => Skip: too few good matches (<{MIN_MATCHES})")
            continue
        # 2) GHT: vote for the position of the model's centroid in the scene for each match
        votes = []  # list of votes (predicted positions of the centroid in the scene)
        for m in good:
            # Model and scene keypoint coordinates
            kp_m = kp_model[m.queryIdx]
            kp_s = kp_scene[m.trainIdx]
            (mx, my) = kp_m.pt
            (sx, sy) = kp_s.pt
            # Vector from model keypoint to model centroid
            v_mx = cx_model - mx
            v_my = cy_model - my
            # Compute orientation and scale difference
            angle_m = kp_m.angle
            angle_s = kp_s.angle
            angle_diff = angle_s - angle_m
            # Use cosine and sine of the angle to rotate the vector
            angle_diff_rad = np.deg2rad(angle_diff)
            cosA = np.cos(angle_diff_rad)
            sinA = np.sin(angle_diff_rad)
            # Rotated vector according to the orientation difference
            v_rot_x = v_mx * cosA - v_my * sinA
            v_rot_y = v_mx * sinA + v_my * cosA
            # Scale the vector based on the keypoint scale ratio
            scale_m = kp_m.size
            scale_s = kp_s.size
            scale_ratio = scale_s / scale_m if scale_m > 1e-6 else 1.0
            v_rot_scaled_x = v_rot_x * scale_ratio
            v_rot_scaled_y = v_rot_y * scale_ratio
            # Predicted position of the centroid in the scene (translate the scene keypoint by this vector)
            cx_pred = sx + v_rot_scaled_x
            cy_pred = sy + v_rot_scaled_y
            votes.append((cx_pred, cy_pred))
        # 3) Clustering of votes to identify multiple instances of the same product
        vote_count = len(votes)
        cluster_indices_list = []
        visited = [False] * vote_count
        for i in range(vote_count):
            if visited[i]:
                continue
            # Start a new cluster with vote i
            visited[i] = True
            cluster_idx = [i]
            # BFS/DFS to find all nearby votes within CLUSTER_EPS
            queue = [i]
            while queue:
                cur = queue.pop(0)
                (cx_cur, cy_cur) = votes[cur]
                for j in range(vote_count):
                    if not visited[j]:
                        (cx_j, cy_j) = votes[j]
                        dx = cx_cur - cx_j
                        dy = cy_cur - cy_j
                        if dx*dx + dy*dy <= CLUSTER_EPS * CLUSTER_EPS:
                            visited[j] = True
                            cluster_idx.append(j)
                            queue.append(j)
            cluster_indices_list.append(cluster_idx)
        # For each cluster of votes found, estimate the product instance
        for ci, cluster_idx in enumerate(cluster_indices_list, start=1):
            cluster_size = len(cluster_idx)
            if cluster_size < 4:
                print(f"  => Skip: cluster {ci} troppo piccolo ({cluster_size} punti)")
                continue
            # 4) Estimating homography with RANSAC for the current cluster
            src_pts = np.float32([kp_model[good[idx].queryIdx].pt for idx in cluster_idx]).reshape(-1,1,2)
            dst_pts = np.float32([kp_scene[good[idx].trainIdx].pt for idx in cluster_idx]).reshape(-1,1,2)
            M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, RANSAC_THRESH)
            if M is None:
                print(f"  => Skip: Homography not found for cluster {ci}")
                continue
            inliers = int(mask.sum())
            print(f"  Cluster {ci}: {cluster_size} match, Inliers RANSAC = {inliers}")
            if inliers < MIN_CLUSTER_INLIERS:
                print(f"  => Skip: few inliers per cluster {ci} (<{MIN_CLUSTER_INLIERS})")
                continue
            # Transform the four corners of the template image according to the homography found
            corners = np.float32([[0,0], [w_model,0], [w_model,h_model], [0,h_model]]).reshape(-1,1,2)
            dst_corners = cv2.perspectiveTransform(corners, M)
            pts = dst_corners.reshape(-1, 2).astype(np.float32)
            # 5) Calculate rotated bounding box of instance
            rot_rect = cv2.minAreaRect(pts)
            (cx, cy), (w_box, h_box), angle = rot_rect
            box_pts = cv2.boxPoints(rot_rect).astype(int)
            # Minimum area filter
            if w_box * h_box < MIN_AREA_RATIO * (w_scene * h_scene):
                print(f"  => Skip: area cluster {ci} too small (<{MIN_AREA_RATIO*100:.2f}% scena)")
                continue
            # 6) Color filter (if necessary for confusing models)
            if mid in CONFUSE_MODELS:
                src_rect = np.float32([[0,0], [w_model,0], [w_model,h_model], [0,h_model]])
                dst_rect = np.float32(box_pts)
                M_inv = cv2.getPerspectiveTransform(dst_rect, src_rect)
                roi = cv2.warpPerspective(img_scene, M_inv, (w_model, h_model))
                mean_h_roi = calc_mean_hue(roi)
                diff_h = abs(mean_h_roi - data['mean_hue'])
                print(f"  Hue ROI={mean_h_roi:.1f}, Model={data['mean_hue']:.1f}, Δ={diff_h:.1f}")
                if diff_h > HUE_DIFF_THRESH:
                    print(f"  => Skip: color difference cluster {ci} too high (>±{HUE_DIFF_THRESH}°)")
                    continue
            # 7) Save detection found for this cluster/instance
            detections.setdefault(mid, []).append({
                'center': (int(round(cx)), int(round(cy))),
                'width':  int(round(w_box)),
                'height': int(round(h_box)),
                'angle':  angle,
                'box_pts': box_pts
            })
            print(f"  *** Detected model instance {mid} (cluster {ci}, {inliers} inliers) ***")
    # Report results for the current scene image
    print(f"\nResults for {scene_file}:")
    if not detections:
        print("  No product recognized.")
    for pid, dets in detections.items():
        print(f"  Model {pid} - {len(dets)} instance(s) found:")
        for idx, det in enumerate(dets, start=1):
            cx, cy = det['center']
            w_b, h_b = det['width'], det['height']
            print(f"    Instance {idx} {{position: ({cx},{cy}), w={w_b}px, h={h_b}px}}")
    # Draw and show results on the scene
    vis = img_scene.copy()
    for pid, dets in detections.items():
        for det in dets:
            # Draw the rotated bounding box
            cv2.drawContours(vis, [det['box_pts']], 0, BOX_COLOR, BOX_THICK)
            # Draw the centroid
            cv2.circle(vis, det['center'], CIRCLE_RAD, CENTER_COLOR, -1)
            # Print the product ID centered with respect to the box
            text = f"ID: {pid}"
            (tw, th), _ = cv2.getTextSize(text, FONT, FONT_SCALE, TEXT_THICK)
            tx = det['center'][0] - tw // 2
            ty = det['center'][1] + th // 2
            cv2.putText(vis, text, (tx, ty), FONT, FONT_SCALE, TEXT_COLOR, TEXT_THICK)
    cv2.imshow(f"Detections - {scene_file}", vis)
    cv2.waitKey(0)
    cv2.destroyAllWindows()



=== Scene Analysis m1.png ===
Kp scena: 4383  Descriptors: (4383, 128)

[Modello 0]: kp_modello=8044  des_modello=(8044, 128)
  Raw matches: 8044
  Good matches (ratio<0.7): 144
  => Skip: cluster 1 troppo piccolo (1 punti)
  => Skip: cluster 2 troppo piccolo (1 punti)
  Cluster 3: 61 match, Inliers RANSAC = 39
  Hue ROI=29.4, Model=48.0, Δ=18.7
  => Skip: color difference cluster 3 too high (>±18°)
  => Skip: cluster 4 troppo piccolo (1 punti)
  => Skip: cluster 5 troppo piccolo (1 punti)
  => Skip: cluster 6 troppo piccolo (1 punti)
  => Skip: cluster 7 troppo piccolo (3 punti)
  => Skip: Homography not found for cluster 8
  => Skip: cluster 9 troppo piccolo (2 punti)
  => Skip: cluster 10 troppo piccolo (1 punti)
  => Skip: cluster 11 troppo piccolo (1 punti)
  => Skip: cluster 12 troppo piccolo (1 punti)
  => Skip: cluster 13 troppo piccolo (1 punti)
  => Skip: cluster 14 troppo piccolo (1 punti)
  => Skip: cluster 15 troppo piccolo (1 punti)
  => Skip: cluster 16 troppo piccolo (